# RIFE desde cero — entrenamiento en Kaggle (2×T4)

Guía completa en `docs/KAGGLE.md`.  Configura **Accelerator: GPU T4 x2** e **Internet: ON**.

Cambia `RESUME = None` por la ruta al `last.pth` de la sesión anterior para reanudar.

In [ ]:
import glob, os

# --- Configuración ---
EPOCHS      = 60        # TOTAL del plan; NO lo cambies entre sesiones
BATCH       = 16        # por GPU
TIME_LIMIT  = 11.2      # horas; para limpiamente antes de las 12 h de Kaggle
RESUME      = None      # p.ej. glob.glob('/kaggle/input/*/checkpoints/last.pth')[0]
SMOKE_FIRST = False     # True → smoke test de 3 min antes del entrenamiento real

# Localiza el dataset (la carpeta que contiene tri_trainlist.txt)
cands = glob.glob('/kaggle/input/**/tri_trainlist.txt', recursive=True)
assert cands, 'No encuentro tri_trainlist.txt: añade el dataset Vimeo90K triplet como Input'
DATA_ROOT = os.path.dirname(cands[0])
OUT = '/kaggle/working/checkpoints'
print('DATA_ROOT =', DATA_ROOT)
print('RESUME    =', RESUME)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!rm -rf /kaggle/working/rife && git clone -q https://github.com/correo415415/super-light-resolution.git /kaggle/working/rife
%cd /kaggle/working/rife
!pip install -q tensorboard 2>/dev/null
!python tests/test_core.py | tail -3

In [ ]:
if SMOKE_FIRST:
    !torchrun --nproc_per_node=2 train.py --data_root {DATA_ROOT} --smoke --amp --num_workers 2 --out_dir /kaggle/working/smoke

In [ ]:
resume_flag = f'--resume {RESUME}' if RESUME else ''
!torchrun --nproc_per_node=2 train.py \
    --data_root {DATA_ROOT} --out_dir {OUT} {resume_flag} \
    --epochs {EPOCHS} --batch_size {BATCH} --amp --num_workers 4 \
    --time_limit {TIME_LIMIT} --save_every 500 --log_every 100

In [ ]:
# Evaluación rápida del mejor checkpoint (500 tríos para no gastar tiempo; quita --max_samples para el test completo)
best = f'{OUT}/best.pth' if os.path.exists(f'{OUT}/best.pth') else f'{OUT}/last.pth'
!python evaluate.py --ckpt {best} --data_root {DATA_ROOT} --amp --max_samples 500 \
    --vis_dir /kaggle/working/vis --n_vis 4 --out_json /kaggle/working/eval.json
!ls -la {OUT}

Al terminar: **Save Version → Quick Save (Save output)**.  En la siguiente sesión añade esta versión como *Input* (Your Work → Notebooks) y pon `RESUME = glob.glob('/kaggle/input/*/checkpoints/last.pth')[0]`.